# JiT-S2-VMamba — complexity of all conditioning arms (params · GMACs · throughput)

Architecture-only profiling: **no checkpoint, no dataset, no training.** Every arm is built from the repo's own `scripts/evaluate.py::build_model` (the same builder that produced the FID numbers), then measured with `src/flops_counter.py::count_complexity` (fvcore + custom hooks for `selective_scan_fn`, SDPA, causal conv).

All arms are measured **in the same session on the same GPU**, so throughput is directly comparable across arms (the numbers printed by the individual eval notebooks came from different Kaggle sessions and possibly different GPUs).

Arms: baseline · stateinit-dimsum · stateinit-learned · ssc-static · ssc-bc · ssc-abc · class-K ∈ {1, 16, 32} · time_class-K ∈ {2, 4} · row-class-U1 · row-time_class-U2.

Runtime: ~5–10 min on a T4. Internet ON (wheels + git clone).

## 1. Environment  *(Internet ON)*

In [ ]:
import os

# 1) Pin torch to 2.5.1
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 \
    --index-url https://download.pytorch.org/whl/cu124

# 2) Download wheels with explicit destination
CAUSAL = "causal_conv1d-1.5.0.post8+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"
MAMBA  = "mamba_ssm-2.2.4+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"

os.system(f"wget -q https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/{CAUSAL} -O /kaggle/working/{CAUSAL}")
os.system(f"wget -q https://github.com/state-spaces/mamba/releases/download/v2.2.4/{MAMBA} -O /kaggle/working/{MAMBA}")

!pip install -q /kaggle/working/{CAUSAL}
!pip install -q /kaggle/working/{MAMBA}

# 3) Patch mamba-ssm
import glob
for path in glob.glob("/usr/local/lib/python*/dist-packages/mamba_ssm/utils/generation.py"):
    with open(path) as f: src = f.read()
    new = src.replace(
        "from transformers.generation import GreedySearchDecoderOnlyOutput, SampleDecoderOnlyOutput, TextStreamer",
        "from transformers.generation import GenerateDecoderOnlyOutput, TextStreamer",
    ).replace(
        "output_cls = GreedySearchDecoderOnlyOutput if top_k == 1 else SampleDecoderOnlyOutput",
        "output_cls = GenerateDecoderOnlyOutput",
    )
    if new != src:
        with open(path, "w") as f: f.write(new)
        print(f"\u2705 Patched {path}")

# 4) fvcore for MAC counting (src/flops_counter.py)
!pip install -q fvcore

print(">>> RESTART RUNTIME NOW <<<")

## 2. Repo  *(clone + cd)*

In [ ]:
import os
REPO_DIR = "/kaggle/working/thesis_Choustoulakis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Rodamanthosch/thesis_Choustoulakis.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log -1 --format='commit %h  %cd  %s'
# sanity: every arm must be wired into evaluate.py's build_model
!grep -q state_init scripts/evaluate.py && grep -q ssc scripts/evaluate.py && grep -q in_context_content scripts/evaluate.py && grep -q in_context_layout scripts/evaluate.py \
    && echo "build_model wiring OK" || echo "WIRING MISSING -- pull latest main"
!grep -q '"static"' src/models/vmamba.py && echo "ssc=static present" || echo "ssc=static MISSING (drop that arm or apply the static patch)"

## 3. Arms  *(one template config + model-key overrides)*

In [ ]:
# All arms share every non-model key; only the conditioning keys differ.
# The template is the state-init config (same S/12 backbone as every arm).
TEMPLATE = "configs/tiny_imagenet/jit-s2-vmamba-stateinit-dimsum.yaml"

OFF = dict(in_context_len=0, in_context_start=4, in_context_content="class",
           in_context_layout="prefix", state_init="none", ssc="none")

def arm(name, **kw):
    m = dict(OFF); m.update(kw); return (name, m)

ARMS = [
    arm("baseline"),
    arm("stateinit-dimsum",  state_init="dimsum"),
    arm("stateinit-learned", state_init="learned"),
    arm("ssc-static", ssc="static"),
    arm("ssc-bc",     ssc="bc"),
    arm("ssc-abc",    ssc="abc"),
    *[arm(f"class-K{k}",      in_context_len=k, in_context_content="class")      for k in (1, 16, 32)],
    *[arm(f"time_class-K{k}", in_context_len=k, in_context_content="time_class") for k in (2, 4)],
    # row layout: the same tokens, one content unit per row/column (grid_size = 8)
    arm("row-class-U1",      in_context_len=8,  in_context_content="class",
                             in_context_layout="row"),
    arm("row-time_class-U2", in_context_len=16, in_context_content="time_class",
                             in_context_layout="row"),
]
for n, m in ARMS:
    print(f"{n:20s}", {k: v for k, v in m.items() if v != OFF[k]})

## 4. Measure  *(params, GMACs, throughput — same GPU for every arm)*

In [ ]:
import sys, copy, gc, time, torch
sys.path.insert(0, REPO_DIR)
from src.utils import load_config
from src.flops_counter import count_complexity
from scripts.evaluate import build_model

assert torch.cuda.is_available(), "needs a GPU (mamba_ssm CUDA kernel)"
DEVICE = "cuda"
GPU = torch.cuda.get_device_name(0)
print("GPU:", GPU)

BATCH, N_WARMUP, N_ITERS = 128, 10, 50

@torch.no_grad()
def throughput(net, m_cfg, grad_mode):
    """img/s for a single forward at batch 128.
    grad_mode=True  reproduces src/utils.measure_throughput (autograd graph is built,
                    as in the numbers printed by evaluate.py);
    grad_mode=False is pure inference (torch.inference_mode)."""
    S, C = m_cfg["input_size"], m_cfg["in_channels"]
    x = torch.randn(BATCH, C, S, S, device=DEVICE)
    t = torch.rand(BATCH, device=DEVICE)
    y = torch.randint(0, m_cfg["num_classes"], (BATCH,), device=DEVICE)
    ctx = torch.enable_grad() if grad_mode else torch.inference_mode()
    with ctx:
        for _ in range(N_WARMUP): net(x, t, y)
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(N_ITERS):  net(x, t, y)
        torch.cuda.synchronize()
    return N_ITERS * BATCH / (time.time() - t0)

base_cfg = load_config(TEMPLATE)
results = []
for name, over in ARMS:
    cfg = copy.deepcopy(base_cfg)
    cfg["experiment"]["name"] = name
    cfg["model"].update(over)
    m_cfg = cfg["model"]
    torch.manual_seed(0)
    net = build_model(cfg).to(DEVICE).eval()

    rep = count_complexity(net, img_size=m_cfg["input_size"], in_channels=m_cfg["in_channels"],
                           num_classes=m_cfg["num_classes"], device=DEVICE)
    ssm_macs = sum(v for k, v in rep["macs_by_op"].items() if "SelectiveScan" in k)
    assert ssm_macs > 0, f"{name}: selective-scan hook did not fire -> MACs undercounted"
    unsup = sorted(k for k in rep["unsupported_ops"] if not any(
        s in k for s in ("aten::add", "aten::mul", "aten::sub", "aten::div", "aten::softplus", "aten::exp",
                         "aten::sigmoid", "aten::silu", "aten::gelu", "aten::neg", "aten::cat", "aten::flip",
                         "aten::clamp", "aten::rsqrt", "aten::pow", "aten::mean", "aten::sin", "aten::cos",
                         "aten::expand", "aten::repeat", "aten::view", "aten::reshape", "aten::permute",
                         "aten::transpose", "aten::contiguous", "aten::split", "aten::chunk", "aten::slice",
                         "aten::unsqueeze", "aten::squeeze", "aten::to", "aten::copy", "aten::zeros",
                         "aten::new_zeros", "aten::arange", "aten::ones", "aten::type_as", "aten::float",
                         "aten::sqrt", "aten::log", "aten::index", "aten::select", "aten::stack", "aten::flatten",
                         "aten::roll", "aten::t", "aten::numel", "aten::size", "aten::Int", "aten::ScalarImplicit",
                         "aten::repeat_interleave", "aten::expm1", "aten::rsub", "aten::einsum")))

    tp_grad = throughput(net, m_cfg, grad_mode=True)
    tp_inf  = throughput(net, m_cfg, grad_mode=False)
    r = dict(arm=name,
             params_M=rep["params_total"] / 1e6,
             gmacs=rep["macs_total"] / 1e9,
             ssm_gmacs=ssm_macs / 1e9,
             tput_evalpy=tp_grad,
             tput_inference=tp_inf,
             unsupported_nontrivial=",".join(unsup))
    results.append(r)
    print(f"{name:20s} params {r['params_M']:6.2f}M | GMACs {r['gmacs']:.3f} (scan {r['ssm_gmacs']:.3f}) | "
          f"img/s {tp_grad:6.1f} (eval.py) {tp_inf:6.1f} (inference)" + (f" | CHECK: {unsup}" if unsup else ""))
    del net; gc.collect(); torch.cuda.empty_cache()

## 5. Table  *(deltas vs baseline, CSV/JSON, LaTeX rows)*

In [ ]:
import pandas as pd, json
df = pd.DataFrame(results).set_index("arm")
b = df.loc["baseline"]
df["d_params_M"] = df["params_M"] - b["params_M"]
df["d_gmacs_%"]  = 100 * (df["gmacs"] / b["gmacs"] - 1)
df["d_tput_%"]   = 100 * (df["tput_inference"] / b["tput_inference"] - 1)
pd.options.display.float_format = "{:.3f}".format
display(df.drop(columns="unsupported_nontrivial"))

OUT = "/kaggle/working/complexity"
os.makedirs(OUT, exist_ok=True)
df.to_csv(f"{OUT}/complexity_{GPU.replace(' ', '_')}.csv")
json.dump({"gpu": GPU, "batch": BATCH, "results": results}, open(f"{OUT}/complexity.json", "w"), indent=2)
print("saved ->", OUT)

print("\n% ---- LaTeX rows: arm & Params (M) & GMACs & Throughput (img/s) ----")
for a, r in df.iterrows():
    print(f"\\texttt{{{a}}} & ${r.params_M:.2f}$ & ${r.gmacs:.3f}$ & ${r.tput_inference:.0f}$ \\\\")

## Notes
- **GMACs** follow the paper convention (fvcore counts one multiply-add; JiT/DiT/VMamba label this "Gflops"). Strict FLOPs = 2 × MACs. Measured for **one image, one forward** at 64×64.
- The selective-scan term uses VMamba's analytical formula (`9·B·L·D·N + B·D·L`); the assert guarantees the hook fired for every arm. A non-empty `CHECK:` list means fvcore met an op it could not count that is not a trivial elementwise/reshape op — inspect it before using that row.
- **Throughput**: `tput_evalpy` reproduces `src/utils.measure_throughput` (autograd enabled, as in the eval logs); `tput_inference` is the clean `inference_mode` number — use this one in the report, and quote the GPU name.
- Complexity does not depend on the weights, so no checkpoint is loaded; `torch.manual_seed(0)` only fixes the random inputs.
- State-init lengthens every scan by 1 position; the in-context prefix by K positions from block `in_context_start` (4) onward; SSC does not change L. The **row** arms lengthen it by the same token count as a prefix of equal size (8 → L=72, 16 → L=80) — only the token *positions* differ, so `row-time_class-U2` should land on the same GMACs as `class-K16`. If it does not, the cost-matching claim in the write-up has to be dropped.